# 2단계 모델 검증: 년도 기준 분할 (훈련 ~2023 / 테스트 2024~2025)

## 1. 라이브러리 임포트

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

## 2. 데이터 불러오기 및 전처리

In [2]:
# 검단 포함 전체 신도시 데이터 불러오기
df = pd.read_csv('real_new_city.csv', encoding='utf-8-sig')

# 거래금액 전처리 
df['거래금액(만원)'] = df['거래금액(만원)'].str.replace(',', '').astype(int)

# m2당가격 파생변수 생성
df['m2당가격'] = df['거래금액(만원)'] / df['전용면적(㎡)']

print(f'전체 데이터: {len(df)}건')

전체 데이터: 126375건


## 3. 이상치 제거

In [3]:
# 전용면적 33제곱미터 미만 제거
df = df[df['전용면적(㎡)'] >= 33]

# 발표후경과년수 3년 미만 필터링
before = len(df)
df = df[df['발표후경과년수'] >= 3]
print(f'발표후경과년수 3 미만 제거: {before - len(df)}개 → 남은 데이터: {len(df)}개')

# m2당가격 z-score 기준 이상치 제거 (|z| > 2인 데이터 제거)
mean = df['m2당가격'].mean()
std = df['m2당가격'].std()
z_scores = (df['m2당가격'] - mean) / std
df = df[z_scores.abs() <= 2]

print(f'이상치 제거 후: {len(df)}건')

발표후경과년수 3 미만 제거: 3863개 → 남은 데이터: 122221개
이상치 제거 후: 116935건


## 4. 년도 기준 훈련/테스트 분할

In [4]:
# 독립변수 목록
features = ['건축년도', '층',
            '지하철호선개수', '기차역까지의거리',
            '가장 가까운 지하철역까지의 거리', '가장 가까운 IC와의 거리',
            '발표후경과년수', 'CPI', '계약연도', '서울도심거리',
            '단지별_세대수', '도시별_세대수']

# 결측치 제거
df = df.dropna(subset=features + ['m2당가격'])

# 년도 기준 분할
df_train = df[df['계약연도'] <= 2023]
df_test  = df[df['계약연도'] >= 2024]

# 독립변수와 타겟 분리
train_input  = df_train[features]
train_target = df_train['m2당가격']
test_input   = df_test[features]
test_target  = df_test['m2당가격']

print(f'훈련 데이터 (~2023): {len(df_train)}건')
print(f'테스트 데이터 (2024~2025): {len(df_test)}건')

훈련 데이터 (~2023): 96811건
테스트 데이터 (2024~2025): 20118건


## 5. 표준화

In [5]:
# 표준화
ss = StandardScaler()
ss.fit(train_input)
train_scaled = ss.transform(train_input)
test_scaled  = ss.transform(test_input)

## 6. 최적 k 탐색

In [6]:
# k 값별로 KNN 모델 학습 및 점수 저장
train_score = []
test_score  = []
k_list = [3, 5, 7, 10, 15, 20, 30, 40, 50]

for k in k_list:
    knn_k = KNeighborsRegressor(n_neighbors=k)
    knn_k.fit(train_scaled, train_target)
    train_score.append(knn_k.score(train_scaled, train_target))
    test_score.append(knn_k.score(test_scaled, test_target))

# 테스트 점수가 가장 높은 k 선택
best_idx = test_score.index(max(test_score))
best_k   = k_list[best_idx]
print(f'최적 k: {best_k}')

최적 k: 7


## 7. 최적 k로 최종 모델 학습

In [7]:
# 최적 k로 KNN 모델 학습
knn_best = KNeighborsRegressor(n_neighbors=best_k)
knn_best.fit(train_scaled, train_target)

print(f"{'='*50}")
print(f'2단계 모델 성능 (년도 기준 분할)')
print(f"{'='*50}")
print(f'최적 k: {best_k}')
print(f'훈련 R²: {knn_best.score(train_scaled, train_target):.4f}')
print(f'테스트 R²: {knn_best.score(test_scaled, test_target):.4f}')

2단계 모델 성능 (년도 기준 분할)
최적 k: 7
훈련 R²: 0.9503
테스트 R²: 0.8229
